# Dashboard analítico del mercado laboral colombiano

Este tablero usa exclusivamente los shards publicados en `pxtron/vacantes-colombia`. Cada resultado representa **publicaciones observadas**, no contrataciones. La revisión exacta de Hugging Face queda registrada al cargar los datos.

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd()
if not (RAIZ / 'src').exists(): RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
from src.analisis.datos_hf import cargar_vacantes_hf
from src.analisis.metricas import (preparar_vacantes, kpis_principales, tabla_completitud, completitud_por_grupo, tabla_frecuencias, resumen_concentracion, diversidad_cargos_region, resumen_salarios, salarios_por_grupo, serie_temporal_mensual)
REPO_ID = 'pxtron/vacantes-colombia'
REVISION = 'main'  # puede fijarse a un commit de HF para reproducibilidad exacta
df_bruto, procedencia = cargar_vacantes_hf(REPO_ID, REVISION)
df = preparar_vacantes(df_bruto)
display(Markdown(f'**Fuente:** `{procedencia.repo_id}` · revisión `{procedencia.revision_resuelta}` · **{len(df):,} publicaciones**'))

## 1. ¿Qué datos tenemos?

In [ ]:
kpis = kpis_principales(df)
fig = go.Figure()
for i, (nombre, valor) in enumerate(list(kpis.items())[:4]):
    fig.add_trace(go.Indicator(mode='number', value=valor, title={'text': nombre.replace('_',' ').title()}, domain={'row': 0, 'column': i}))
fig.update_layout(grid={'rows': 1, 'columns': 4, 'pattern': 'independent'}, height=220, title='Cobertura general')
fig.show()
display(Markdown('Las tarjetas describen el alcance del corpus; empresas corresponde a organizaciones o prestadores observados, no a contrataciones verificadas.'))

In [ ]:
calidad = tabla_completitud(df)
display(calidad.style.format({'cobertura_pct': '{:.1f}%'}))
fig = px.bar(calidad, x='cobertura_pct', y='campo', orientation='h', range_x=[0,100], title='Completitud por campo (%)')
fig.show()
calidad_fuente = completitud_por_grupo(df)
px.imshow(calidad_fuente, text_auto='.0f', aspect='auto', color_continuous_scale='Blues', zmin=0, zmax=100, title='Completitud por campo y portal (%)').show()
display(Markdown('La completitud define qué preguntas pueden responderse y revela diferencias entre fuentes. En particular, un valor ausente de modalidad no equivale a trabajo presencial.'))

In [ ]:
fuentes = tabla_frecuencias(df, 'portal')
display(fuentes.style.format({'participacion_pct':'{:.2f}%'}))
px.bar(fuentes, x='portal', y='publicaciones', text_auto=True, title='Participación de cada portal').show()
display(Markdown('Las comparaciones agregadas deben leerse junto con esta composición: una fuente dominante puede determinar gran parte del patrón observado.'))

## 2. ¿Dónde están las oportunidades?

In [ ]:
dept = tabla_frecuencias(df, 'departamento', 15)
ciudad = tabla_frecuencias(df, 'ciudad', 15)
px.bar(dept.sort_values('publicaciones'), x='publicaciones', y='departamento', orientation='h', title='Departamentos con más publicaciones').show()
px.bar(ciudad.sort_values('publicaciones'), x='publicaciones', y='ciudad', orientation='h', title='Ciudades con más publicaciones').show()
display(pd.DataFrame([resumen_concentracion(df, 'departamento'), resumen_concentracion(df, 'ciudad')]).style.format({'hhi':'{:.3f}', 'top_1_pct':'{:.1f}%', 'top_5_pct':'{:.1f}%'}))
display(diversidad_cargos_region(df, minimo=100).head(15).style.format({'diversidad_normalizada':'{:.3f}'}))
display(Markdown('El HHI y la participación de los cinco territorios principales permiten evaluar concentración, no solo ordenar conteos. No se dibuja un mapa porque el esquema actual no incluye códigos DIVIPOLA.'))

## 3. ¿Qué perfiles y condiciones se buscan?

In [ ]:
for col, titulo in [('titulo_normalizado','Cargos con mayor demanda'), ('tipo_contrato','Tipos de contrato'), ('nivel_educativo','Nivel educativo requerido'), ('modalidad_categoria','Modalidad informada')]:
    tabla = tabla_frecuencias(df, col, 15)
    px.bar(tabla.sort_values('publicaciones'), x='publicaciones', y=col, orientation='h', title=titulo).show()
empresas = tabla_frecuencias(df, 'empresa', 15)
px.bar(empresas.sort_values('publicaciones'), x='publicaciones', y='empresa', orientation='h', title='Organizaciones o prestadores con más publicaciones').show()
display(pd.Series(resumen_concentracion(df, 'empresa')).to_frame('valor'))
display(Markdown('Los cargos solo agrupan variantes ortográficas seguras. Las organizaciones representan publicaciones observadas y pueden ser intermediarios; no equivalen a contrataciones. En modalidad, desconocida no se convierte en presencial.'))

In [ ]:
exp = df['experiencia_meses'].dropna()
px.histogram(exp, nbins=30, title='Experiencia solicitada (meses)', labels={'value':'meses'}).show()
print(f'Cobertura de experiencia válida: {len(exp)/len(df):.1%}')
print(f'Entre registros válidos, exige experiencia: {df.loc[df.experiencia_meses.notna(), "requiere_experiencia"].mean():.1%}')
display(Markdown('La proporción que exige experiencia usa solamente registros interpretables; valores superiores a 40 años se consideran inválidos y no se usan.'))

## 4. ¿Cómo evoluciona la publicación?

In [ ]:
temporal = serie_temporal_mensual(df)
px.line(temporal, x='periodo', y='publicaciones', markers=True, title='Publicaciones por mes').show()
display(temporal.tail(12).style.format({'variacion_pct':'{:+.1f}%'}))
display(Markdown('La variación describe fechas de publicación, no nuevas capturas. Los meses extremos se marcan como incompletos y no deben usarse para concluir crecimiento o caída.'))

## 5. ¿Cuánto se paga?

In [ ]:
salarios = resumen_salarios(df)
display(salarios.to_frame('valor').style.format('{:,.0f}'))
s = df.loc[df.salario_comparable, 'salario_representativo'].dropna()
if len(s):
    limite = s.quantile(.99)
    px.histogram(s[s <= limite], nbins=40, title='Punto medio de rangos salariales (hasta p99)', labels={'value':'COP mensuales'}).show()
display(Markdown('La gráfica limita la vista al percentil 99, pero la tabla conserva los extremos. Se analizan puntos medios de rangos nominales, no salarios efectivamente pagados.'))

In [ ]:
for col in ['departamento','nivel_educativo','tipo_contrato','experiencia_tramo','titulo_normalizado']:
    tabla = salarios_por_grupo(df, col, minimo=30)
    if len(tabla):
        display(Markdown(f'### Salario por {col.replace("_"," ")}'))
        display(tabla.head(20).style.format({'mediana':'{:,.0f}', 'media':'{:,.0f}', 'q1':'{:,.0f}', 'q3':'{:,.0f}'}))
grupos_edu = salarios_por_grupo(df, 'nivel_educativo', minimo=30)['nivel_educativo'].tolist()
base_box = df[df.salario_comparable & df.nivel_educativo.isin(grupos_edu)].copy()
if len(base_box):
    base_box = base_box[base_box.salario_representativo <= base_box.salario_representativo.quantile(.99)]
    px.box(base_box, x='nivel_educativo', y='salario_representativo', points=False, title='Dispersión salarial por nivel educativo (hasta p99)').show()
display(Markdown('Solo aparecen grupos con al menos 30 salarios comparables. Las diferencias son asociaciones descriptivas y no efectos causales de educación, contrato o territorio.'))

## Lectura responsable

Este dashboard caracteriza publicaciones disponibles en las fuentes. No mide contratación efectiva, personas empleadas, salarios finales ni causalidad. Las tendencias dependen de la cobertura temporal y de la composición por portal.